<a href="https://colab.research.google.com/github/Prktriesitout/Langchain_Projects/blob/main/Project4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**AI Job Interview Coach**

In the modern job market, candidates often face challenges in preparing for interviews in a structured, confident, and professional manner. HR teams and career counselors seek an AI-powered Interview Coach that can simulate realistic interview scenarios, evaluate candidate responses, and provide constructive feedback.

The goal is to build a Job Interview Coaching Assistant using LangChain that leverages Prompt Templates and Few-Shot Examples. By exposing the AI to examples of strong and weak responses, the assistant can guide learners toward delivering structured, impactful, and professional answers.

Design and implement an AI Interview Coach using LangChain that:

Utilizes a Prompt Template to enforce structured responses (e.g., introduction, main answer, feedback).
Incorporates Few-Shot Examples to demonstrate what constitutes strong versus weak interview answers.
Simulates realistic interview questions across multiple domains (e.g., software engineering, data science, business analysis).
Provides personalized, step-by-step feedback to help candidates improve their performance.

Step 1: Load an Open-Source LLM

Use a Hugging Face model (e.g., tiiuae/falcon-7b-instruct or google/flan-t5-base) integrated via LangChain.

In [4]:
!pip install -U --force-reinstall langchain-core langchain-huggingface


  Using cached langchain_core-1.8.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached langchain_huggingface-1.2.3-py3-none-any.whl.metadata (4.3 kB)
Using cached langchain_core-1.8.1-py3-none-any.whl (588 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 940.8/940.8 kB 10.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 29.9 MB/s  0:00:00
Using cached langchain_huggingface-1.2.3-py3-none-any.whl (34 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 32.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 65.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 24.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 79.4 MB/s  0:00:00
  Attempting uninstall: zstandard
    Found existing installation: zstandard 0.25.0
    Uninstalling zstandard-0.25.0:
      Successfully uninstalled zstandard-0.25.0
  Attempting uninstall: xxha

In [8]:
!pip install -U langchain-core langchain-cohere cohere


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 27.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [langchain-cohere]


In [9]:
import os
from google.colab import userdata
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_cohere import ChatCohere

# --------------------------
# 1. Load Cohere API Key from Secrets ("cohere")
# --------------------------
os.environ["COHERE_API_KEY"] = userdata.get("cohere")

# --------------------------
# 2. Initialize Cohere Model
# --------------------------
llm = ChatCohere(
    model="command-a-03-2025",  # Cohere's flagship model (or use "command-r-plus-08-2024")
    temperature=0.3,
    max_tokens=500,
)



Step 2: Define a Prompt Template

Create a template with placeholders such as:

{job_role} → target role for the interview
{interview_question} → question asked by the interviewer
{candidate_response} → candidate’s answer to be evaluated

In [10]:
examples = [
    {
        "interview_question": "Tell me about yourself.",
        "candidate_response": "I am John. I like coding. I need a job.",
        "feedback": "Weak answer. It lacks structure and doesn't highlight professional achievements. Instead, start with your background, key skills , and end how it aligns with the job."
    },
    {
        "interview_question": "Tell me about yourself.",
        "candidate_response": "I am John, a software engineer with 5 years of experience in backend development using Python and cloud technologies. I enjoy solving real-world problems with scalable systems, and I am excited to apply these skills to contribute to your compamy's innovative projects.",
        "feedback": "Strong answer. It is structured, professional, and connects personal skills with the role."
    },
    {
        "interview_question": "Why do you want to join our commpany?",
        "candidate_response": "I admire your company;s focus on innovation in AI and data-driven solutions. I belive my background in machine learning aligns with your mission and I'm eager to contribute to impactful projects that improve customer experiences.",
        "feedback": "Strong answer. It is structured, professional, and connects personal skills with the role."
    }
]

Step 3: Add Few-Shot Examples

Provide 2–3 examples of interview questions with both weak and strong answers.
Demonstrate how structured feedback should be presented.


In [11]:
example_template = """
Interview Question: {interview_question}
Candidate Response: {candidate_response}
Feedback: {feedback}
"""

example_prompt = PromptTemplate(
    input_variables=["interview_question", "candidate_response", "feedback"],
    template=example_template
)

overall_template = """
You are an AI Interview Coach.
Evaluate the candidate's answer and provide structured feedback.

Follow this format:
1. Strengths
2. Improvements
3. Final Recommendation (Strong / Weak)

Examples: {{examples}}
Interview Question: {interview_question}
Candidate Response: {candidate_response}
Feedback:
"""



Step 4: Build the Chain

Use FewShotPromptTemplate in LangChain to combine examples with the dynamic template.

In [12]:
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix=overall_template,
    input_variables=["interview_question", "candidate_response"],
)
chain = prompt | llm | StrOutputParser()

In [13]:
from rich import print
print(chain)


RunnableSequence(
    first=FewShotPromptTemplate(
        input_variables=['candidate_response', 'interview_question'],
        input_types={},
        partial_variables={},
        examples=[
            {
                'interview_question': 'Tell me about yourself.',
                'candidate_response': 'I am John. I like coding. I need a job.',
                'feedback': "Weak answer. It lacks structure and doesn't highlight professional achievements. 
Instead, start with your background, key skills , and end how it aligns with the job."
            },
            {
                'interview_question': 'Tell me about yourself.',
                'candidate_response': "I am John, a software engineer with 5 years of experience in backend 
development using Python and cloud technologies. I enjoy solving real-world problems with scalable systems, and I 
am excited to apply these skills to contribute to your compamy's innovative projects.",
                'feedback': 'Strong answer. It is structured, professional, and connects personal skills with the 
role.'
            },
            {
                'interview_question': 'Why do you want to join our commpany?',
                'candidate_response': "I admire your company;s focus on innovation in AI and data-driven solutions.
I belive my background in machine learning aligns with your mission and I'm eager to contribute to impactful 
projects that improve customer experiences.",
                'feedback': 'Strong answer. It is structured, professional, and connects personal skills with the 
role.'
            }
        ],
        example_prompt=PromptTemplate(
            input_variables=['candidate_response', 'feedback', 'interview_question'],
            input_types={},
            partial_variables={},
            template='\nInterview Question: {interview_question}\nCandidate Response: 
{candidate_response}\nFeedback: {feedback}\n'
        ),
        suffix="\nYou are an AI Interview Coach.\nEvaluate the candidate's answer and provide structured 
feedback.\n\nFollow this format:\n1. Strengths \n2. Improvements \n3. Final Recommendation (Strong / 
Weak)\n\nExamples: {{examples}}\nInterview Question: {interview_question}\nCandidate Response: 
{candidate_response}\nFeedback:\n"
    ),
    middle=[
        ChatCohere(
            client=<cohere.client.Client object at 0x7ae5cf15dd30>,
            async_client=<cohere.client.AsyncClient object at 0x7ae5cf15e3c0>,
            model='command-a-03-2025',
            temperature=0.3,
            cohere_api_key=SecretStr('**********'),
            metadata={'lc_versions': {'langchain-core': '1.8.1', 'langchain': '1.4.8'}}
        )
    ],
    last=StrOutputParser()
)

Step 5: Simulate Interview Q&A

User provides a job role and candidate response.

The system:

Reframes the candidate’s answer.
Compares it against strong examples.
Provides structured feedback, including strengths, areas for improvement, and final recommendations.

In [14]:
response = chain.invoke({
    "interview_question":"Explain a challenging project you worked on.",
    "candidate_response": "I led a project migrating legacy databases to cloud, improving data preprocessing speed by 40%. I faced performance issues intially but resolved them with query optimization and parallelization."
})

print(response)

**Interview Question:** Explain a challenging project you worked on.  
**Candidate Response:** I led a project migrating legacy databases to cloud, improving data preprocessing speed by 
40%. I faced performance issues initially but resolved them with query optimization and parallelization.  

**Feedback:**  

**1. Strengths:**  
- **Quantifiable Impact:** The candidate highlights a specific achievement (40% improvement in data preprocessing 
speed), which demonstrates measurable results.  
- **Problem-Solving:** Clearly articulates a challenge (performance issues) and the steps taken to overcome it 
(query optimization and parallelization).  
- **Leadership:** Mentions leading the project, which showcases initiative and responsibility.  

**2. Improvements:**  
- **Context:** The candidate could provide more context about the project, such as the scale of the migration or 
the industry it impacted.  
- **Collaboration:** Mentioning teamwork or how they coordinated with others would strengthen the response, as most
projects involve collaboration.  
- **Learning:** Adding a reflection on what they learned from the experience would add depth to the answer.  

**3. Final Recommendation:** **Strong**. The response is clear, structured, and highlights key achievements. With 
minor improvements, it could be exceptional.

Step 6: Test with Sample Questions Example questions:

“Tell me about yourself.”
“Why do you want to join our company?”
“Explain a challenging project you worked on.”

In [16]:
response = chain.invoke({
    "interview_question":"Tell me about yourself.",
    "candidate_response": "My name is Pratyush Kumar , i have completed my Bachleors in Electronics and communication engineering , I have also worked on a research paper 'Fabrication of V2O5 schottky diode ', which got selected for the final exhibition of the college , I soon started working in infosys where i learned about Agentic Enablement concepts such as langchain , langgraph , multimodalities in AI , i also did a full course on Data Science from upgrad where i learned various algorithms , currently i am interning for An AI - ML company , primarily i make agents train models , At present i was working on a tool where i reduced the API charges from 5$ to 1$ per tool call"

})

print(response)

**Feedback:**

1. **Strengths:**  
   - The response provides a clear educational background (Bachelor’s in Electronics and Communication 
Engineering).  
   - Highlights relevant technical skills and experiences (research paper, work at Infosys, Data Science course, 
and current internship).  
   - Demonstrates measurable impact (reducing API charges from $5 to $1 per tool call).  
   - Shows progression from academia to industry, with a focus on AI/ML.  

2. **Improvements:**  
   - **Structure:** The answer could be more organized. Start with a brief introduction, followed by key skills, 
professional experiences, and how they align with the role.  
   - **Relevance:** Tailor the response to the job being applied for. For example, explicitly connect the research 
paper, Infosys experience, or internship to the role’s requirements.  
   - **Clarity:** Avoid jargon or abbreviations (e.g., "Agentic Enablement") unless the interviewer is familiar 
with them.  
   - **Flow:** The transition between points feels abrupt. Use connecting phrases to make the narrative smoother.  

3. **Final Recommendation:** **Strong, but could be refined.**  
   The candidate demonstrates a solid background and relevant achievements, but improving structure and tailoring 
the response to the role would make it even stronger.  

**Revised Example Response (for reference):**  
"My name is Pratyush Kumar, and I hold a Bachelor’s degree in Electronics and Communication Engineering. During my 
studies, I worked on a research paper titled 'Fabrication of V2O5 Schottky Diode,' which was selected for the final
exhibition at my college. This experience sparked my interest in technology and problem-solving.  

I soon joined Infosys, where I gained hands-on experience with AI concepts like LangChain, LangGraph, and 
multimodalities. I also completed a comprehensive Data Science course from UpGrad, where I honed my skills in 
various machine learning algorithms.  

Currently, I’m interning at an AI-ML company, where I focus on training models and building intelligent agents. 
Recently, I developed a tool that reduced API charges from $5 to $1 per call, significantly optimizing costs.  

I’m excited about this role because my background in AI/ML and my passion for creating efficient, scalable 
solutions align perfectly with your company’s goals. I’m eager to contribute to your innovative projects and drive 
impactful results."

Step 7: Evaluation & Enhancement

Ensure the system maintains a professional, coaching-oriented tone.
Test across different roles (e.g., Software Engineer, Data Analyst, Business Analyst).
Extend to multi-turn sessions where the AI recalls prior candidate responses.